# Add Gaia XP Derivative, Uncertainty, and SNR Features

This notebook extends the baseline `c110` coefficient dataset with derivative coefficients (`d110`), coefficient-uncertainty, signal-to-noise, and XP-summary quality features. Existing packaged CSV outputs are reused when present.


## Imports

In [ ]:
import io
import os
import time
import random
import ast
import re
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import requests
from astropy.io.votable import parse_single_table



## Configuration

In [ ]:
ROOT = Path.cwd().resolve()
OUT_DIR = ROOT / "out_data"
OUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_CSV = OUT_DIR / "gaia_dr3_xp_c110_l2_binary.csv"
DERIVATIVE_CSV = OUT_DIR / "gaia_dr3_xp_c110_d110_l2_binary.csv"
FULL_FEATURE_CSV = OUT_DIR / "gaia_dr3_xp_c110_d110_errors_snr_binary.csv"

SOURCE_ID_COL = "source_id"
LABEL_COL = "y"
TAP_URL = "https://gaia.aip.de/tap"
SJS_URL = "https://gaia.aip.de/uws/simple-join-service"

SAMPLING = np.linspace(0.0, 60.0, 600)
RIDGE_LAM = 1e-3
DERIVATIVE_ID_CHUNK_SIZE = 500
ERROR_ID_CHUNK_SIZE = 300


print("ROOT            :", ROOT)
print("BASE_CSV        :", BASE_CSV)
print("DERIVATIVE_CSV  :", DERIVATIVE_CSV)
print("FULL_FEATURE_CSV:", FULL_FEATURE_CSV)

## Gaia@AIP Authentication Helpers

In [ ]:
PLACEHOLDER_TOKENS = {"", "YOUR_TOKEN_GOES_HERE", "<YOUR_TOKEN>", "<TOKEN>", "TOKEN"}


def load_dotenv_if_available() -> None:
    try:
        from dotenv import load_dotenv
    except Exception:
        return
    load_dotenv(ROOT / ".env")
    load_dotenv()


def normalize_gaia_aip_token(raw_token: str) -> str:
    token = (raw_token or "").strip()
    if len(token) >= 2 and token[0] in {"'", '"'} and token[-1] == token[0]:
        token = token[1:-1].strip()
    if token.upper() in PLACEHOLDER_TOKENS:
        raise RuntimeError(
            "Gaia@AIP token is missing. Set GAIA_AIP_TOKEN, define it in .env, "
            "or create .gaia_aip_token with your Gaia@AIP API token."
        )
    if any(ord(char) > 127 for char in token):
        raise RuntimeError("GAIA_AIP_TOKEN must contain ASCII characters only. Check copied whitespace or quotes.")
    return token if token.startswith("Token ") else f"Token {token}"


def make_gaia_aip_session() -> requests.Session:
    load_dotenv_if_available()
    raw_token = os.getenv("GAIA_AIP_TOKEN", "").strip()
    token_file = Path(os.getenv("GAIA_AIP_TOKEN_FILE", str(ROOT / ".gaia_aip_token"))).expanduser()
    if not raw_token and token_file.exists():
        raw_token = token_file.read_text(encoding="utf-8").strip()
    token = normalize_gaia_aip_token(raw_token)
    session = requests.Session()
    session.headers["Authorization"] = token
    return session


def raise_for_gaia_aip_response(response: requests.Response, context: str) -> None:
    text = response.text or ""
    if response.status_code in {401, 403}:
        raise RuntimeError(
            f"Gaia@AIP rejected {context} with HTTP {response.status_code}. "
            "Check GAIA_AIP_TOKEN. The value may be either the raw token or 'Token <token>'. "
            "Restart the kernel after changing credentials."
        )
    response.raise_for_status()

## Load the Baseline Dataset

In [ ]:
if not BASE_CSV.exists():
    raise FileNotFoundError(
        f"Missing baseline coefficient dataset: {BASE_CSV}. "
        "Run 01_build_gaia_xp_coefficient_dataset.ipynb first."
    )

df_cls = pd.read_csv(BASE_CSV)
required = {SOURCE_ID_COL, LABEL_COL, "c000", "c109"}
missing = required - set(df_cls.columns)
if missing:
    raise KeyError(f"{BASE_CSV} is missing required columns: {sorted(missing)}")

df_cls[SOURCE_ID_COL] = pd.to_numeric(df_cls[SOURCE_ID_COL], errors="raise").astype("int64")
df_cls[LABEL_COL] = pd.to_numeric(df_cls[LABEL_COL], errors="raise").astype(int)
source_ids = np.sort(df_cls[SOURCE_ID_COL].unique())

print("Baseline shape:", df_cls.shape)
print("Unique source IDs:", len(source_ids))
print("Class balance:", df_cls[LABEL_COL].value_counts().sort_index().to_dict())

## Derivative Feature Helpers

In [ ]:
def tap_async_start(
    session: requests.Session,
    query: str,
    *,
    lang: str = "ADQL",
    runid: str = "xp_ids",
    max_retries: int = 5,
    backoff_s: float = 2.0,
):
    url = f"{TAP_URL}/async"
    q = query.strip().rstrip(";")
    payload = {"REQUEST":"doQuery", "LANG":lang, "FORMAT":"votable", "QUERY":q, "RUNID":runid}

    last_err = None
    for attempt in range(max_retries + 1):
        try:
            r = session.post(url, data=payload, allow_redirects=False, timeout=120)
        except requests.RequestException as e:
            last_err = e
            if attempt < max_retries:
                sleep_s = backoff_s * (2 ** attempt)
                sleep_s *= 0.5 + random.random()  # jitter
                time.sleep(sleep_s)
                continue
            raise

        if r.status_code in (302, 303) and "Location" in r.headers:
            job_url = r.headers["Location"]
            if job_url.startswith("/"):
                job_url = "https://gaia.aip.de" + job_url
            session.post(f"{job_url}/phase", data={"PHASE":"RUN"}, timeout=60).raise_for_status()
            return job_url

        # retry on transient gateway/load errors
        if r.status_code in (429, 500, 502, 503, 504):
            last_err = RuntimeError(f"TAP async start failed: HTTP {r.status_code}; body={r.text[:300]!r}")
            if attempt < max_retries:
                sleep_s = backoff_s * (2 ** attempt)
                sleep_s *= 0.5 + random.random()
                time.sleep(sleep_s)
                continue

        raise RuntimeError(f"TAP async start failed: HTTP {r.status_code}; body={r.text[:300]!r}")

    raise last_err if last_err is not None else RuntimeError("TAP async start failed (unknown).")

def tap_async_wait(session: requests.Session, job_url: str, *, timeout_s: int = 240):
    t0 = time.time()
    while True:
        ph = session.get(f"{job_url}/phase", timeout=60).text.strip()
        if ph in ("COMPLETED", "ERROR", "ABORTED"):
            break
        if time.time() - t0 > timeout_s:
            raise TimeoutError(f"TAP async timed out (> {timeout_s}s)")
        time.sleep(1.5)

    if ph != "COMPLETED":
        err = session.get(f"{job_url}/error", timeout=60).text if ph == "ERROR" else ""
        raise RuntimeError(f"TAP async ended with phase={ph}. {err[:800]}")
    return ph

def tap_async_download_votable(session: requests.Session, job_url: str) -> pd.DataFrame:
    res_url = f"{job_url}/results/result"
    content = session.get(res_url, timeout=300).content
    tab = parse_single_table(io.BytesIO(content)).to_table(use_names_over_ids=True)
    return tab.to_pandas()


def to_1d_float(x, expected_len: Optional[int] = None) -> np.ndarray:
    # If the column is a masked array, drop the mask and keep numeric data
    if isinstance(x, np.ma.MaskedArray):
        x = np.ma.getdata(x)

    # Parse string representations if TAP returns arrays as strings
    if isinstance(x, str):
        s = x.strip()
        try:
            x = ast.literal_eval(s)
        except Exception:
            s = s.strip("[]()")
            parts = re.split(r"[,\s]+", s)
            parts = [p for p in parts if p]
            x = [float(p) for p in parts]

    arr = np.asarray(x, dtype=float).reshape(-1)
    if expected_len is not None and arr.size != expected_len:
        raise ValueError(f"Expected len={expected_len}, got {arr.size}")
    return arr


def build_design_matrices_via_convert(bp_basis_id: int, rp_basis_id: int, sampling: np.ndarray):

    n = 55
    I = np.eye(n, dtype=float)

    # Minimal columns for gaiaxpy input validation (kept lightweight)
    common = {
        "bp_n_parameters": n,
        "bp_standard_deviation": 1.0,
        "rp_n_parameters": n,
        "rp_standard_deviation": 1.0,
        "bp_coefficient_errors": [np.ones(n, dtype=float) for _ in range(n)],
        "rp_coefficient_errors": [np.ones(n, dtype=float) for _ in range(n)],
        "bp_coefficient_correlations": [np.eye(n, dtype=float) for _ in range(n)],
        "rp_coefficient_correlations": [np.eye(n, dtype=float) for _ in range(n)],
    }

    # BP units
    df_bp = pd.DataFrame({
        "source_id": np.arange(n, dtype=np.int64),
        "bp_basis_function_id": int(bp_basis_id),
        "rp_basis_function_id": int(rp_basis_id),
        "bp_coefficients": [I[i] for i in range(n)],
        "rp_coefficients": [np.zeros(n, dtype=float) for _ in range(n)],
        **common,
    })
    df_bp_samp, u = convert(df_bp, sampling=sampling, truncation=False, with_correlation=False, save_file=False)
    df_bp_samp = df_bp_samp[df_bp_samp["xp"].str.upper() == "BP"].sort_values("source_id")
    bp_fluxes = np.vstack([to_1d_float(x) for x in df_bp_samp["flux"]])  # (55,m)
    A_BP = bp_fluxes.T  # (m,55)

    # RP units
    df_rp = pd.DataFrame({
        "source_id": np.arange(n, dtype=np.int64),
        "bp_basis_function_id": int(bp_basis_id),
        "rp_basis_function_id": int(rp_basis_id),
        "bp_coefficients": [np.zeros(n, dtype=float) for _ in range(n)],
        "rp_coefficients": [I[i] for i in range(n)],
        **common,
    })
    df_rp_samp, u2 = convert(df_rp, sampling=sampling, truncation=False, with_correlation=False, save_file=False)
    df_rp_samp = df_rp_samp[df_rp_samp["xp"].str.upper() == "RP"].sort_values("source_id")
    rp_fluxes = np.vstack([to_1d_float(x) for x in df_rp_samp["flux"]])  # (55,m)
    A_RP = rp_fluxes.T

    if not np.allclose(u, u2):
        raise RuntimeError("Sampling grids differ between BP/RP (unexpected).")

    return A_BP.astype(float), A_RP.astype(float), np.asarray(u, float)


def ridge_projector(A: np.ndarray, lam: float) -> np.ndarray:
    # A: (m,55) -> returns P: (55,m)
    n = A.shape[1]
    return np.linalg.solve(A.T @ A + lam*np.eye(n), A.T)

def batch_flux(C: np.ndarray, A: np.ndarray) -> np.ndarray:
    # C: (N,55), A: (m,55) -> flux: (N,m)
    return C @ A.T

def batch_derivative(F: np.ndarray, u: np.ndarray) -> np.ndarray:
    # F: (N,m)
    return np.gradient(F, u, axis=1)

def l2_normalize_rows(X: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    n = np.linalg.norm(X, axis=1, keepdims=True)
    n = np.maximum(n, eps)
    return X / n


def _fetch_xp_for_ids(ids: list[int]) -> pd.DataFrame:
    ids_sql = ",".join(str(int(x)) for x in ids)
    q = f'''
    SELECT
      source_id,
      bp_basis_function_id, rp_basis_function_id,
      bp_coefficients, rp_coefficients
    FROM gaiadr3.xp_continuous_mean_spectrum
    WHERE source_id IN ({ids_sql})
    '''
    job_url = tap_async_start(sess, q, runid="xp_ids_chunk")
    tap_async_wait(sess, job_url, timeout_s=300)
    return tap_async_download_votable(sess, job_url)

def fetch_xp_for_ids(ids, *, min_chunk: int = 200, max_split_depth: int = 6) -> pd.DataFrame:
    ids_list = [int(x) for x in ids]
    try:
        return _fetch_xp_for_ids(ids_list)
    except RuntimeError as e:
        msg = str(e)
        transient = ("HTTP 429", "HTTP 500", "HTTP 502", "HTTP 503", "HTTP 504", "timed out")
        if any(t in msg for t in transient) and max_split_depth > 0 and len(ids_list) > min_chunk:
            mid = len(ids_list) // 2
            left = fetch_xp_for_ids(ids_list[:mid], min_chunk=min_chunk, max_split_depth=max_split_depth - 1)
            right = fetch_xp_for_ids(ids_list[mid:], min_chunk=min_chunk, max_split_depth=max_split_depth - 1)
            return pd.concat([left, right], ignore_index=True)
        raise

## Build or Load the Derivative Dataset

In [ ]:
if DERIVATIVE_CSV.exists():
    df_merged = pd.read_csv(DERIVATIVE_CSV)
    print("Using existing derivative dataset:", DERIVATIVE_CSV)
    print("Shape:", df_merged.shape)
else:
    print("Derivative dataset not found; rebuilding from Gaia@AIP:", DERIVATIVE_CSV)
    sess = make_gaia_aip_session()
    d_cols = [f"d{i:03d}" for i in range(110)]
    CACHE = {}  # (bp_basis_id, rp_basis_id) -> (A_BP, A_RP, u, P_BP, P_RP)

    def process_df_to_der110(df: pd.DataFrame) -> pd.DataFrame:
        outs = []
        for (bp_id, rp_id), g in df.groupby(["bp_basis_function_id", "rp_basis_function_id"], sort=False):
            key = (int(bp_id), int(rp_id))
            if key not in CACHE:
                A_BP, A_RP, u = build_design_matrices_via_convert(key[0], key[1], SAMPLING)
                P_BP = ridge_projector(A_BP, RIDGE_LAM)
                P_RP = ridge_projector(A_RP, RIDGE_LAM)
                CACHE[key] = (A_BP, A_RP, u, P_BP, P_RP)

            A_BP, A_RP, u, P_BP, P_RP = CACHE[key]

            sids = g["source_id"].astype(np.int64).to_numpy()
            C_bp = np.vstack([to_1d_float(x, 55) for x in g["bp_coefficients"]])
            C_rp = np.vstack([to_1d_float(x, 55) for x in g["rp_coefficients"]])

            F_bp = batch_flux(C_bp, A_BP)
            F_rp = batch_flux(C_rp, A_RP)

            d_bp = batch_derivative(F_bp, u)
            d_rp = batch_derivative(F_rp, u)

            Cbp_prime = d_bp @ P_BP.T  # (N,55)
            Crp_prime = d_rp @ P_RP.T  # (N,55)

            D = np.hstack([Cbp_prime, Crp_prime])          # (N,110)
            D = l2_normalize_rows(D).astype(np.float32)    # L2-normalized features

            out = pd.DataFrame({SOURCE_ID_COL: sids})
            out[d_cols] = D
            outs.append(out)

        if not outs:
            return pd.DataFrame(columns=[SOURCE_ID_COL] + d_cols)

        out_all = pd.concat(outs, ignore_index=True)
        out_all = out_all.drop_duplicates(SOURCE_ID_COL)
        return out_all

    parts = []
    total_found = 0

    for i0 in range(0, len(source_ids), ID_CHUNK_SIZE):
        chunk = source_ids[i0:i0 + ID_CHUNK_SIZE]
        print(f"Chunk {i0//ID_CHUNK_SIZE+1}: ids={len(chunk)}")

        df_xp = fetch_xp_for_ids(chunk)
        if len(df_xp) == 0:
            print("  No XP rows returned for this chunk (no XP available for these source_ids).")
            continue

        df_der110 = process_df_to_der110(df_xp)
        parts.append(df_der110)
        total_found += len(df_der110)
        print("  Derivative feature rows:", len(df_der110), "| total so far:", total_found)

    if not parts:
        raise RuntimeError("No XP continuous rows returned for any chunk. Check token / classification IDs / availability of XP.")

    df_der110_all = pd.concat(parts, ignore_index=True).drop_duplicates(SOURCE_ID_COL)
    print("Total derivative feature rows:", len(df_der110_all))

    # Merge into the classification CSV (keep all original rows)
    df_merged = df_cls.merge(df_der110_all, on=SOURCE_ID_COL, how="left")

    coverage = df_merged[d_cols[0]].notna().mean() * 100.0
    print(f"Coverage: {coverage:.1f}% of classification rows got derivative features")

    df_merged.to_csv(DERIVATIVE_CSV, index=False)
    print("Saved derivative dataset:", DERIVATIVE_CSV)

    df_merged.head()

## Build or Load Error, SNR, and XP-Summary Features

In [ ]:
if FULL_FEATURE_CSV.exists():
    df_full = pd.read_csv(FULL_FEATURE_CSV)
    print("Using existing full feature dataset:", FULL_FEATURE_CSV)
    print("Shape:", df_full.shape)
else:
    print("Full feature dataset not found; rebuilding from Gaia@AIP:", FULL_FEATURE_CSV)

    sess = make_gaia_aip_session()
    # 3) Helpers (TAP async + SJS)
    def read_first_votable(path: Path) -> pd.DataFrame:
        tab = parse_single_table(str(path)).to_table(use_names_over_ids=True)
        return tab.to_pandas()

    def tap_create_async_job(session: requests.Session, query: str, *, lang: str = "ADQL", runid: str = "ids_for_sjs") -> str:
        url = f"{TAP_URL}/async"
        query = query.strip().rstrip(";")
        payload = {
            "REQUEST": "doQuery",
            "LANG": lang,
            "FORMAT": "votable",
            "QUERY": query,
            "RUNID": runid,
        }
        r = session.post(url, data=payload, allow_redirects=False, timeout=120)

        if r.status_code == 403 and "Invalid token" in (r.text or ""):
            raise RuntimeError("HTTP 403: Invalid Gaia@AIP token. Check GAIA_AIP_TOKEN and restart the kernel.")

        if r.status_code in (302, 303) and "Location" in r.headers:
            job_url = r.headers["Location"]
            if job_url.startswith("/"):
                job_url = "https://gaia.aip.de" + job_url
        else:
            raise RuntimeError(f"Unexpected TAP async response: HTTP {r.status_code}; body={r.text[:300]!r}")

        job_id = job_url.rstrip("/").split("/")[-1]
        session.post(f"{job_url}/phase", data={"PHASE": "RUN"}, timeout=60).raise_for_status()

        t0 = time.time()
        while True:
            ph = session.get(f"{job_url}/phase", timeout=60).text.strip()
            if ph in ("COMPLETED", "ERROR", "ABORTED"):
                break
            if time.time() - t0 > 180:
                raise TimeoutError("TAP async job timed out (>180s).")
            time.sleep(1.5)

        if ph != "COMPLETED":
            raise RuntimeError(f"TAP async job ended with phase={ph}")

        return job_id

    def tap_async_query_to_df(session: requests.Session, query: str, *, runid: str = "tap_query") -> pd.DataFrame:
        # create
        url = f"{TAP_URL}/async"
        q = query.strip().rstrip(";")
        payload = {"REQUEST":"doQuery","LANG":"ADQL","FORMAT":"votable","QUERY":q,"RUNID":runid}
        r = session.post(url, data=payload, allow_redirects=False, timeout=120)

        if r.status_code == 403 and "Invalid token" in (r.text or ""):
            raise RuntimeError("HTTP 403: Invalid Gaia@AIP token. Check GAIA_AIP_TOKEN and restart the kernel.")

        if r.status_code in (302, 303) and "Location" in r.headers:
            job_url = r.headers["Location"]
            if job_url.startswith("/"):
                job_url = "https://gaia.aip.de" + job_url
        else:
            raise RuntimeError(f"Unexpected TAP async response: HTTP {r.status_code}; body={r.text[:300]!r}")

        # run
        session.post(f"{job_url}/phase", data={"PHASE":"RUN"}, timeout=60).raise_for_status()

        t0 = time.time()
        while True:
            ph = session.get(f"{job_url}/phase", timeout=60).text.strip()
            if ph in ("COMPLETED","ERROR","ABORTED"):
                break
            if time.time() - t0 > 240:
                raise TimeoutError("TAP async job timed out (>240s).")
            time.sleep(1.5)

        if ph != "COMPLETED":
            raise RuntimeError(f"TAP async job ended with phase={ph}")

        # download
        res_url = f"{job_url}/results/result"
        content = session.get(res_url, timeout=300).content
        tab = parse_single_table(io.BytesIO(content)).to_table(use_names_over_ids=True)
        return tab.to_pandas()

    def sjs_download(session: requests.Session, job_id: str, join_table: str, *,
                     column_name: str = "source_id",
                     responseformat: str = "votable",
                     data_structure: str = "COMBINED") -> Path:
        service = vo.dal.DALService(SJS_URL, session=session)
        q = service.create_query(
            job_id=job_id,
            column_name=column_name,
            responseformat=responseformat,
            join_table=join_table,
            data_structure=data_structure,
        )
        resp = q.submit(post=True)
        resp.raise_for_status()
        job = vo.io.uws.parse_job(io.BytesIO(resp.content))

        service._session.post(f"{service._baseurl}/{job.jobid}/phase", data={"PHASE": "RUN"}, stream=True).raise_for_status()

        t0 = time.time()
        while True:
            ph = service._session.get(f"{service._baseurl}/{job.jobid}/phase").text.strip()
            if ph in ("COMPLETED", "ERROR", "ABORTED"):
                break
            if time.time() - t0 > 240:
                raise TimeoutError("SJS job timed out (>240s).")
            time.sleep(1.5)

        if ph != "COMPLETED":
            raise RuntimeError(f"SJS ended with phase={ph}")

        job_url = f"{service._baseurl}/{job.jobid}"
        job2 = vo.io.uws.parse_job(io.BytesIO(service._session.get(job_url).content))

        results = job2.results
        if hasattr(results, "keys") and callable(getattr(results, "keys")):
            first_key = sorted(list(results.keys()))[0]
            href = results[first_key].href
            key = str(first_key)
        else:
            res_list = list(results)
            if not res_list:
                raise RuntimeError("SJS job has no results.")
            href = res_list[0].href
            key = "result"

        out_path = OUT_DIR / f"tmp_sjs_{job2.jobid}_{key}.vot"

        last = None
        for attempt in range(1, 5):
            r = service._session.get(href, timeout=300)
            r.raise_for_status()
            content = r.content
            last = content

            # basic sanity checks for a VOTable payload
            ok = (content is not None) and (len(content) > 500) and (b"VOTABLE" in content[:5000]) and (b"</VOTABLE>" in content[-2000:])
            if ok:
                out_path.write_bytes(content)
                return out_path

            time.sleep(1.0 * attempt)

        out_path.write_bytes(last or b"")
        raise RuntimeError(
            "Downloaded SJS result looks truncated / not a valid VOTable. "
            f"Saved payload to: {out_path}. Try re-running the cell."
        )


    # 4) Fetch XP continuous arrays (coefficients + coefficient_errors) via SJS
    ID_CHUNK_SIZE = 300

    def fetch_xp_continuous_for_ids(ids: np.ndarray) -> pd.DataFrame:
        ids_sql = ",".join(str(int(sid)) for sid in ids)
        q_ids_job = f"""
        SELECT source_id
        FROM gaiadr3.gaia_source
        WHERE source_id IN ({ids_sql})
        """
        tap_job_id = tap_create_async_job(sess, q_ids_job, runid="ids_for_sjs")
        path_vot = sjs_download(
            sess,
            tap_job_id,
            "gaiadr3.xp_continuous_mean_spectrum",
            responseformat="votable",
            data_structure="COMBINED",
        )
        df = read_first_votable(path_vot)
        keep = ["source_id",
                "bp_coefficients", "bp_coefficient_errors",
                "rp_coefficients", "rp_coefficient_errors"]
        keep = [c for c in keep if c in df.columns]
        return df[keep].copy()

    dfs = []
    for i0 in range(0, len(source_ids), ID_CHUNK_SIZE):
        chunk = source_ids[i0:i0+ID_CHUNK_SIZE]
        print(f"SJS chunk {i0//ID_CHUNK_SIZE+1}: ids={len(chunk)}")
        df_xp = fetch_xp_continuous_for_ids(chunk)
        print("  rows:", len(df_xp))
        if len(df_xp):
            dfs.append(df_xp)

    df_xp_all = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame(columns=["source_id"])
    df_xp_all["source_id"] = pd.to_numeric(df_xp_all["source_id"], errors="coerce").astype("Int64")
    df_xp_all = df_xp_all.dropna(subset=["source_id"]).copy()
    df_xp_all["source_id"] = df_xp_all["source_id"].astype("int64")
    df_xp_all = df_xp_all.drop_duplicates("source_id")

    print("XP continuous rows fetched:", len(df_xp_all), "out of", len(source_ids))
    df_xp_all.head()


    # 5) Fetch xp_summary quality indicators via TAP (scalars)
    XP_SUMMARY_COLS = [
        "bp_n_relevant_bases", "bp_relative_shrinking", "bp_n_measurements", "bp_n_rejected_measurements",
        "bp_standard_deviation", "bp_chi_squared", "bp_n_transits", "bp_n_contaminated_transits", "bp_n_blended_transits",
        "rp_n_relevant_bases", "rp_relative_shrinking", "rp_n_measurements", "rp_n_rejected_measurements",
        "rp_standard_deviation", "rp_chi_squared", "rp_n_transits", "rp_n_contaminated_transits", "rp_n_blended_transits",
    ]

    def fetch_xp_summary_for_ids(ids: np.ndarray) -> pd.DataFrame:
        ids_sql = ",".join(str(int(sid)) for sid in ids)
        cols_sql = ",\n  ".join(["source_id"] + XP_SUMMARY_COLS)
        q = f"""
        SELECT
          {cols_sql}
        FROM gaiadr3.xp_summary
        WHERE source_id IN ({ids_sql})
        """
        return tap_async_query_to_df(sess, q, runid="xp_summary")

    dfs = []
    for i0 in range(0, len(source_ids), ID_CHUNK_SIZE):
        chunk = source_ids[i0:i0+ID_CHUNK_SIZE]
        print(f"TAP xp_summary chunk {i0//ID_CHUNK_SIZE+1}: ids={len(chunk)}")
        df_s = fetch_xp_summary_for_ids(chunk)
        print("  rows:", len(df_s))
        if len(df_s):
            dfs.append(df_s)

    df_sum = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame(columns=["source_id"])
    df_sum["source_id"] = pd.to_numeric(df_sum["source_id"], errors="coerce").astype("Int64")
    df_sum = df_sum.dropna(subset=["source_id"]).copy()
    df_sum["source_id"] = df_sum["source_id"].astype("int64")
    df_sum = df_sum.drop_duplicates("source_id")

    print("xp_summary rows fetched:", len(df_sum), "out of", len(source_ids))
    df_sum.head()


    # 6) Build features: errors (110) + SNR (110) + summary stats
    def _to_float_array(x):
        # handle numpy arrays, masked arrays, lists
        if x is None:
            return None
        try:
            arr = np.asarray(x, dtype=float)
            return arr
        except Exception:
            try:
                return np.asarray(list(x), dtype=float)
            except Exception:
                return None

    def expand_array_column(df: pd.DataFrame, col: str, prefix: str, expected_len: int) -> pd.DataFrame:
        arr2d = np.full((len(df), expected_len), np.nan, dtype=float)
        for i, x in enumerate(df[col].values):
            a = _to_float_array(x)
            if a is None or a.ndim != 1 or len(a) != expected_len:
                continue
            arr2d[i, :] = a
        cols = [f"{prefix}_{k:02d}" for k in range(expected_len)]
        return pd.DataFrame(arr2d, columns=cols)

    # infer expected lengths (usually 55)
    def infer_len(series):
        for x in series.values:
            a = _to_float_array(x)
            if a is not None and a.ndim == 1:
                return int(len(a))
        return None

    n_bp = infer_len(df_xp_all.get("bp_coefficients", pd.Series([], dtype=object)))
    n_rp = infer_len(df_xp_all.get("rp_coefficients", pd.Series([], dtype=object)))

    if n_bp is None or n_rp is None:
        raise RuntimeError("Could not infer coefficient vector length from xp_continuous_mean_spectrum.")
    print("BP coeff len:", n_bp, "| RP coeff len:", n_rp)

    # Expand errors
    df_bp_err = expand_array_column(df_xp_all, "bp_coefficient_errors", "bp_err", expected_len=n_bp)
    df_rp_err = expand_array_column(df_xp_all, "rp_coefficient_errors", "rp_err", expected_len=n_rp)

    # Expand coefficients (for SNR)
    df_bp_c = expand_array_column(df_xp_all, "bp_coefficients", "bp_c", expected_len=n_bp)
    df_rp_c = expand_array_column(df_xp_all, "rp_coefficients", "rp_c", expected_len=n_rp)

    # Compute per-coefficient SNR
    eps = 1e-12
    bp_snr = np.abs(df_bp_c.to_numpy()) / (df_bp_err.to_numpy() + eps)
    rp_snr = np.abs(df_rp_c.to_numpy()) / (df_rp_err.to_numpy() + eps)

    df_bp_snr = pd.DataFrame(bp_snr, columns=[f"bp_snr_{k:02d}" for k in range(n_bp)])
    df_rp_snr = pd.DataFrame(rp_snr, columns=[f"rp_snr_{k:02d}" for k in range(n_rp)])

    # Summary stats
    def summarize(arr2d: np.ndarray, prefix: str) -> pd.DataFrame:
        return pd.DataFrame({
            f"{prefix}_mean": np.nanmean(arr2d, axis=1),
            f"{prefix}_std":  np.nanstd(arr2d, axis=1),
            f"{prefix}_min":  np.nanmin(arr2d, axis=1),
            f"{prefix}_max":  np.nanmax(arr2d, axis=1),
            f"{prefix}_median": np.nanmedian(arr2d, axis=1),
        })

    df_stats = pd.concat([
        summarize(df_bp_err.to_numpy(), "bp_err"),
        summarize(df_rp_err.to_numpy(), "rp_err"),
        summarize(bp_snr, "bp_snr"),
        summarize(rp_snr, "rp_snr"),
    ], axis=1)

    # Assemble per-source feature table
    df_feat = pd.concat([
        df_xp_all[["source_id"]].reset_index(drop=True),
        df_bp_err.reset_index(drop=True),
        df_rp_err.reset_index(drop=True),
        df_bp_snr.reset_index(drop=True),
        df_rp_snr.reset_index(drop=True),
        df_stats.reset_index(drop=True),
    ], axis=1)

    print("Feature table rows:", len(df_feat), "| cols:", df_feat.shape[1])
    df_feat.head()


    # Merge everything into a duplicate CSV
    # Prefix xp_summary columns to avoid collisions
    df_sum2 = df_sum.copy()
    for c in XP_SUMMARY_COLS:
        if c in df_sum2.columns:
            df_sum2.rename(columns={c: f"xps_{c}"}, inplace=True)

    df_merged = df_base.merge(df_feat, on="source_id", how="left")
    df_merged = df_merged.merge(df_sum2[["source_id"] + [c for c in df_sum2.columns if c.startswith("xps_")]],
                                on="source_id", how="left")

    # Coverage report
    cov_xp = df_merged["bp_err_00"].notna().mean() if "bp_err_00" in df_merged.columns else 0.0
    cov_sum = df_merged.get("xps_bp_n_transits", pd.Series([np.nan]*len(df_merged))).notna().mean()

    print(f"Coverage: xp_continuous errors present for {cov_xp*100:.1f}% rows")
    print(f"Coverage: xp_summary present for {cov_sum*100:.1f}% rows")

    # Output path
    out_name = INPUT_CSV.stem + "_errors_snr_xpsummary.csv"
    OUT_CSV = FULL_FEATURE_CSV

    df_merged.to_csv(OUT_CSV, index=False)
    print("Saved:", OUT_CSV)
    df_merged.head()

## Final Validation

In [ ]:
final_path = FULL_FEATURE_CSV if FULL_FEATURE_CSV.exists() else DERIVATIVE_CSV
final_df = pd.read_csv(final_path)

required_groups = {
    "c": [f"c{i:03d}" for i in range(110)],
    "d": [f"d{i:03d}" for i in range(110)],
}
for group, columns in required_groups.items():
    missing = [column for column in columns if column not in final_df.columns]
    if missing:
        raise KeyError(f"Final dataset is missing {group}-feature columns: {missing[:8]}")

bp_err = [column for column in final_df.columns if re.fullmatch(r"bp_err_\d{2}", column)]
rp_err = [column for column in final_df.columns if re.fullmatch(r"rp_err_\d{2}", column)]
bp_snr = [column for column in final_df.columns if re.fullmatch(r"bp_snr_\d{2}", column)]
rp_snr = [column for column in final_df.columns if re.fullmatch(r"rp_snr_\d{2}", column)]

print("Validated dataset:", final_path)
print("Shape:", final_df.shape)
print("Feature groups:", {"c": 110, "d": 110, "bp_err": len(bp_err), "rp_err": len(rp_err), "bp_snr": len(bp_snr), "rp_snr": len(rp_snr)})
final_df.head()